In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("login_behavior_dataset.csv")

In [3]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly
0,user_1,2026-04-28 02:09:32,Australia,Mac,Edge,13,Failed,1
1,user_1,2026-05-20 13:28:55,UK,iPhone,Safari,0,Success,0
2,user_1,2026-05-04 17:39:55,UK,iPhone,Safari,3,Success,0
3,user_1,2026-05-06 10:59:42,UK,iPhone,Safari,0,Success,0
4,user_1,2026-05-16 13:34:28,UK,iPhone,Safari,1,Success,0


In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [5]:
df["hour"] = df["timestamp"].dt.hour

In [6]:
df["day_of_week"] = df["timestamp"].dt.dayofweek

In [7]:
df["is_weekend"] = df["day_of_week"].apply(
    lambda x: 1 if x >= 5 else 0
)

In [8]:
df = df.sort_values(
    by=["user_id", "timestamp"]
)

In [9]:
user_baselines = df.groupby("user_id").agg({

    "country": lambda x: x.mode()[0],

    "device_type": lambda x: x.mode()[0],

    "browser": lambda x: x.mode()[0],

    "failed_attempts": "max",
    
    "hour": ["min", "max"]

}).reset_index()

In [19]:
user_baselines.columns = [

    "user_id",

    "usual_country",

    "usual_device",

    "usual_browser",

    "max_normal_failed_attempts",

    "usual_start_hour",

    "usual_end_hour",

]

In [20]:
user_baselines.head()

,user_id,usual_country,usual_device,usual_browser,max_normal_failed_attempts,usual_start_hour,usual_end_hour
0,user_1,UK,iPhone,Safari,16,0,23
1,user_10,USA,Android,Safari,20,1,23
2,user_100,Canada,Linux,Firefox,19,0,23
3,user_11,UK,Windows,Chrome,17,0,18
4,user_12,India,Windows,Chrome,16,0,23


In [21]:
df = df.merge(
    user_baselines,
    on="user_id",
    how="left"
)

In [22]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,...,usual_browser_x,max_normal_failed_attempts_x,usual_start_hour_x,usual_end_hour_x,usual_country_y,usual_device_y,usual_browser_y,max_normal_failed_attempts_y,usual_start_hour_y,usual_end_hour_y
0,user_1,2026-04-26 13:43:59,UK,iPhone,Safari,1,Success,0,13,6,...,Safari,16,0,23,UK,iPhone,Safari,16,0,23
1,user_1,2026-04-26 20:15:15,UK,iPhone,Safari,3,Success,0,20,6,...,Safari,16,0,23,UK,iPhone,Safari,16,0,23
2,user_1,2026-04-27 15:38:08,UK,iPhone,Safari,1,Success,0,15,0,...,Safari,16,0,23,UK,iPhone,Safari,16,0,23
3,user_1,2026-04-27 21:01:05,UK,iPhone,Safari,1,Success,0,21,0,...,Safari,16,0,23,UK,iPhone,Safari,16,0,23
4,user_1,2026-04-27 21:37:38,UK,iPhone,Safari,0,Success,0,21,0,...,Safari,16,0,23,UK,iPhone,Safari,16,0,23


In [23]:
df = df.loc[:, ~df.columns.str.endswith('_y')]

In [24]:
df.columns = df.columns.str.replace('_x', '')

In [25]:
df.head()

,user_id,timestamp,country,device_type,browser,failed_attempts,login_status,is_anomaly,hour,day_of_week,is_weekend,usual_country,usual_device,usual_browser,max_normal_failed_attempts,usual_start_hour,usual_end_hour
0,user_1,2026-04-26 13:43:59,UK,iPhone,Safari,1,Success,0,13,6,1,UK,iPhone,Safari,16,0,23
1,user_1,2026-04-26 20:15:15,UK,iPhone,Safari,3,Success,0,20,6,1,UK,iPhone,Safari,16,0,23
2,user_1,2026-04-27 15:38:08,UK,iPhone,Safari,1,Success,0,15,0,0,UK,iPhone,Safari,16,0,23
3,user_1,2026-04-27 21:01:05,UK,iPhone,Safari,1,Success,0,21,0,0,UK,iPhone,Safari,16,0,23
4,user_1,2026-04-27 21:37:38,UK,iPhone,Safari,0,Success,0,21,0,0,UK,iPhone,Safari,16,0,23


In [26]:
df.to_csv(
    "behavior_preprocessed_data.csv",
    index=False
)

print("Behavioral Preprocessing Complete!")

Behavioral Preprocessing Complete!
